# 📘 Cuaderno maestro — Modelos Estadísticos · G43
**Diplomado en Técnicas Estadísticas y Minería de Datos | UNAM, FES Acatlán**  
**Módulo II · Profesora: Act. Claudia Cristina Reyes Montes de Oca**  
**Sesiones documentadas:** 7 y 11 de noviembre de 2026  
**Estado:** Cuaderno vivo — versión 0.1

> **Origen del contenido:** el apartado de probabilidad laplaciana se basa en el notebook oficial `Proba_Laplaciana_G43_7Nov26.ipynb`. Los ejemplos de probabilidad condicional, dos dados y variables aleatorias proceden de las capturas de clase compartidas el 11/11. Las explicaciones adicionales, visualizaciones y ejercicios son elaboración de apoyo, no transcripción literal de la profesora.

**Cómo usarlo:** ejecuta las celdas en orden (Kernel → Restart & Run All). Cada sección contiene intuición, teoría, Python y una comprobación. Agrega nuevas sesiones al final y conserva el historial en GitHub.

## Índice
1. Preparación del entorno
2. Espacio muestral y probabilidad laplaciana (7 nov)
3. Probabilidad condicional (11 nov)
4. Suma de dos dados (11 nov)
5. Variables aleatorias y transformaciones (11 nov)
6. Ejercicios de práctica y bitácora

### Convenciones
- $\Omega$: espacio muestral; $|A|$: cardinalidad de un conjunto.
- $P(A\mid B)$: probabilidad de $A$ **dado** $B$.
- `A & B`: intersección; `len(A)`: cardinalidad; `Fraction`: fracción exacta.

## 1. Preparación del entorno
Usaremos únicamente la biblioteca estándar de Python para los cálculos principales. `matplotlib` es opcional para las gráficas.

In [ ]:
from itertools import product
from fractions import Fraction
from collections import Counter

print("Entorno preparado: itertools, fractions y collections")

## 2. Espacio muestral y probabilidad laplaciana · 7 nov
**Intuición.** Un experimento aleatorio tiene varios resultados posibles. El espacio muestral $\Omega$ los reúne todos; un evento $A$ es un subconjunto de $\Omega$.

Si los resultados elementales son **equiprobables**, entonces:

$$P(A)=\frac{|A|}{|\Omega|}.$$

**Ejemplo oficial adaptado:** lanzar una moneda tres veces, con `A` = águila y `S` = sol.

In [ ]:
Omega_monedas = set(product(('A', 'S'), repeat=3))
print('Espacio muestral:', sorted(Omega_monedas))
print('Total de resultados:', len(Omega_monedas))

### Eventos e intersección
Definimos los eventos de obtener águila en el primero, segundo y tercer lanzamiento. La intersección exige tres águilas.

In [ ]:
A1 = {om for om in Omega_monedas if om[0] == 'A'}
A2 = {om for om in Omega_monedas if om[1] == 'A'}
A3 = {om for om in Omega_monedas if om[2] == 'A'}
tres_aguilas = A1 & A2 & A3
P_tres_aguilas = Fraction(len(tres_aguilas), len(Omega_monedas))
print('Evento:', tres_aguilas)
print('Probabilidad exacta:', P_tres_aguilas, '| Decimal:', float(P_tres_aguilas))
assert P_tres_aguilas == Fraction(1, 8)

### Función reutilizable de probabilidad laplaciana
En el cuaderno oficial aparece una función que divide cardinalidades. Aquí añadimos validaciones para evitar usarla incorrectamente.

In [ ]:
def prob_laplaciana(evento, omega):
    if not omega:
        raise ValueError('El espacio muestral no puede estar vacío.')
    if not evento <= omega:
        raise ValueError('El evento debe ser subconjunto del espacio muestral.')
    return Fraction(len(evento), len(omega))

print('P(tres águilas) =', prob_laplaciana(tres_aguilas, Omega_monedas))

## 3. Probabilidad condicional · 11 nov
**Intuición.** Saber que ocurrió $C$ reduce el conjunto de resultados que debemos considerar.

$$P(B\mid C)=\frac{P(B\cap C)}{P(C)}=\frac{|B\cap C|}{|C|},\qquad P(C)>0.$$

La segunda igualdad requiere resultados elementales equiprobables.

**Ejercicio visto en clase:**
- $B$: el primer lanzamiento es águila.
- $C$: exactamente dos de los tres lanzamientos son águila.
- Calcular $P(B\mid C)$.

In [ ]:
B = {om for om in Omega_monedas if om[0] == 'A'}
C = {om for om in Omega_monedas if om.count('A') == 2}

def prob_cond(E, F):
    if not F:
        raise ValueError('El evento condicionante debe tener probabilidad positiva.')
    return Fraction(len(E & F), len(F))

print('B:', sorted(B))
print('C:', sorted(C))
print('B ∩ C:', sorted(B & C))
print('P(B | C):', prob_cond(B, C))
print('P(C | B):', prob_cond(C, B))
assert prob_cond(B, C) == Fraction(2, 3)
assert prob_cond(C, B) == Fraction(1, 2)

**Interpretación:** entre las tres secuencias con exactamente dos águilas, dos comienzan con águila. Por eso $P(B\mid C)=2/3$. **Atención:** $P(B\mid C)\ne P(C\mid B)$ en general.

### Pregunta de comprobación
¿Por qué el denominador de $P(B\mid C)$ es 3 y no 8? Responde antes de continuar.

## 4. Dos dados: eventos definidos por la suma · 11 nov
Al lanzar dos dados distinguibles, el espacio muestral tiene $6\times 6=36$ pares ordenados. Definimos

$$S_n=\{(i,j)\in\Omega: i+j=n\}.$$

La profesora creó una función `S(n)` con comprensión de conjuntos. Aquí la conservamos conceptualmente.

In [ ]:
Omega_dados = set(product(range(1, 7), repeat=2))

def S(n):
    return {(i, j) for i in range(1, 7) for j in range(1, 7) if i + j == n}

print('S(7):', sorted(S(7)))
print('P(suma = 7):', prob_laplaciana(S(7), Omega_dados))
assert len(Omega_dados) == 36
assert len(S(7)) == 6

### Tabla de distribución de la suma
Los 36 pares son equiprobables, pero las sumas de 2 a 12 **no** lo son.

In [ ]:
distribucion = {n: prob_laplaciana(S(n), Omega_dados) for n in range(2, 13)}
for suma, prob in distribucion.items():
    print(f'Suma {suma:2d}: {len(S(suma))} casos | P = {str(prob):>4} = {float(prob):.2%}')
assert sum(distribucion.values()) == 1

In [ ]:
try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(9, 4))
    plt.bar(list(distribucion), [float(v) for v in distribucion.values()])
    plt.xticks(range(2, 13))
    plt.xlabel('Suma de los dos dados')
    plt.ylabel('Probabilidad')
    plt.title('Distribución de la suma de dos dados honestos')
    plt.tight_layout()
    plt.show()
except ImportError:
    print('Gráfica opcional: instala matplotlib con %pip install matplotlib')

## 5. Variables aleatorias y transformaciones · 11 nov
**Intuición.** Una variable aleatoria asigna un número a cada resultado del experimento:

$$X:\Omega\longrightarrow\mathbb R.$$

En clase se dibujó un dado honesto con valores del 1 al 6 y, después, la **estatura de estudiantes** como ejemplo de variable numérica.

> **Precisión matemática adicional:** en la definición general se exige que $X$ sea medible. En estos ejemplos finitos con todos los subconjuntos medibles, esta condición se cumple automáticamente.

Si $X$ es una variable aleatoria y $a,c$ son constantes reales, podemos definir otra variable aleatoria $Y=aX+c$. En la pizarra apareció $2X+c$.

In [ ]:
Omega_estudiantes = {'est_1', 'est_2', 'est_3', 'est_4'}
# Valores inventados solo para ilustrar la transformación; NO son datos reales de la clase.
X = {'est_1': 150, 'est_2': 160, 'est_3': 170, 'est_4': 175}
c = 10
Y = {estudiante: 2 * estatura + c for estudiante, estatura in X.items()}
print('Estaturas X:', X)
print('Transformación Y = 2X + 10:', Y)

### Ampliación teórica (no mostrada en las capturas)
Si existen esperanza y varianza:

$$\mathbb E[aX+c]=a\mathbb E[X]+c,\qquad \mathrm{Var}(aX+c)=a^2\mathrm{Var}(X).$$

Estas propiedades se estudiarán con mayor profundidad cuando corresponda.

## 6. Ejercicios para practicar
1. Con tres monedas, calcula $P(\text{exactamente una águila})$.
2. Calcula $P(\text{suma de dados}=8)$.
3. Calcula $P(\text{primer dado}=3\mid\text{suma}=7)$.
4. Explica con palabras la diferencia entre $P(B\mid C)$ y $P(C\mid B)$.

Ejecuta la celda de abajo para verificar los tres resultados numéricos.

In [ ]:
ej1 = {om for om in Omega_monedas if om.count('A') == 1}
ej2 = S(8)
ej3 = {(i, j) for (i, j) in Omega_dados if i == 3}
print('Ejercicio 1:', prob_laplaciana(ej1, Omega_monedas))
print('Ejercicio 2:', prob_laplaciana(ej2, Omega_dados))
print('Ejercicio 3:', prob_cond(ej3, S(7)))
assert prob_laplaciana(ej1, Omega_monedas) == Fraction(3, 8)
assert prob_laplaciana(ej2, Omega_dados) == Fraction(5, 36)
assert prob_cond(ej3, S(7)) == Fraction(1, 6)

## Bitácora de actualizaciones
| Versión | Fecha | Cambios |
|---|---|---|
| 0.1 | 11/11/2026 | Material de 7 nov y capturas de 11 nov: Laplace, condicional, dados y variables aleatorias. |

### Próxima actualización
Agregar el tema que continúe en clase, con fecha, fuente, intuición, desarrollo matemático, código y al menos un ejercicio. **No sustituir ni sobrescribir los notebooks oficiales.**

### Publicación en GitHub
Nombre de repositorio sugerido: `diplomado-temd-g43`. Ruta sugerida: `modulo-02-modelos-estadisticos/Cuaderno_Maestro_Modelos_Estadisticos_G43.ipynb`. No publicar datos privados, credenciales ni información sensible.